In [0]:
%sql
CREATE CATALOG IF NOT EXISTS f1_project;
CREATE SCHEMA IF NOT EXISTS f1_project.default;

In [0]:
import requests
import json
from pyspark.sql.functions import current_timestamp, lit, from_json, schema_of_json, col

# 1. Parameter Widget (Default to 'current')
dbutils.widgets.text("season", "current", "Season Year or 'current'")
season_param = dbutils.widgets.get("season")

# 2. Ensure Catalog & Schema exist
spark.sql("CREATE CATALOG IF NOT EXISTS f1_project")
spark.sql("CREATE SCHEMA IF NOT EXISTS f1_project.default")

# 3. Paginate through API pages (Limit max is 100 per call)
LIMIT = 100
offset = 0
total = 1  # Initial boundary
json_pages = []
headers = {"User-Agent": "F1DataEngineeringPipeline/1.0"}

while offset < total:
    url = f"https://api.jolpi.ca/ergast/f1/{season_param}/results.json?limit={LIMIT}&offset={offset}"
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        raise Exception(f"API request failed at offset {offset} for season '{season_param}' with status: {response.status_code}")
    
    payload = response.json()
    
    # Read total records available from API metadata
    total = int(payload["MRData"]["total"])
    
    # Store page payload as JSON string
    json_pages.append(json.dumps(payload))
    
    # Advance to next page
    offset += LIMIT

print(f"Successfully fetched {len(json_pages)} page(s) covering {total} total result records for season '{season_param}'.")

# 4. Create PySpark DataFrame from all page payloads
df_str = spark.createDataFrame([(p,) for p in json_pages], ["raw_json"])

# 5. Infer dynamic schema from the first page sample
sample_json = json_pages[0]
json_schema = spark.range(1).select(schema_of_json(sample_json)).collect()[0][0]

# 6. Parse JSON payloads and append audit metadata
df_bronze = (
    df_str
    .select(from_json(col("raw_json"), json_schema).alias("parsed"))
    .select("parsed.*")
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_api", lit("Jolpica_F1"))
    .withColumn("requested_season", lit(season_param))
)

# 7. Append all pages to Bronze Delta Table
(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("f1_project.default.f1_project_data_bronze")
)

In [0]:
%sql
select * from f1_project.default.f1_project_data_bronze